<h1 style="text-align: center;"> Project name </h1>

<div style="display: flex; justify-content: space-around;">

<div style="width: 30%; text-align: center;">
<strong>Jie Zhao</strong>  
<br>
jiz273@g.harvard.edu
</div>

<div style="text-align: center; width: 80%; margin: 0 auto;">
    <strong>Abstract</strong><br>
   This project presents a conversational AI analyst for Boston Food Establishment Inspection data using Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), and text-to-SQL techniques. Public inspection datasets contain structured information such as violation codes, severity codes, inspection dates, and business information, as well as unstructured textual comments that are difficult for non-technical users to analyze.
The proposed system allows users to ask natural-language questions about inspection trends, repeated violations, severity patterns, and violation meanings without requiring SQL or data analysis expertise. The architecture combines deterministic SQL analytics for structured numerical queries with semantic retrieval and summarization for contextual inspection comments. The system integrates vector embeddings, semantic search, and tool-calling workflows to provide analytical insights and conversational responses.
This project demonstrates how transformer-based language models and RAG pipelines can improve accessibility and understanding of public health inspection data through an interactive and domain-specific AI analytical system.

</div>

## Table of Contents

**1. [Problem Statement](#problem-statement)**  

**2. [Installation, Configuration and Set UP](#introduction)**  

**3. [Dataset and Data Preparation](#comprehensive-eda-review)**  

**4. [Step-by-Step Project Development]**  
     4.1 set up duck DB

     4.2 Set up SQL function

     4.3 Set up Chart function

     4.4 Test function

     4.5 Deep learning and Embeding

     4.6 Route Function

     4.7 LLM

**5. [Results and Demonstration](#baseline-model)**  

**6. [Uses and Benefits](#final-model)**  

**7. [Challenges / Lesson Learnt]

**8. [Youtube Video Links] 

**9. [References] 

**10. [Appendix] 



 ## 1. Problem Statement

The Health Division of the Department of Inspectional Services conducts inspections on food establishments in the City of Boston to ensure compliance with sanitary codes and food safety regulations. High-risk establishments often require repeated and follow-up inspections. These inspection activities generate large amounts of public data, including violation descriptions, inspection results, dates, locations, severity levels, and business information.


Despite the availability of this data through public datasets, the information remains difficult for non-technical users to explore and analyze effectively. Extracting meaningful insights typically requires knowledge of SQL, data analytics, or visualization tools. As a result, restaurant owners, city analysts, public health teams, and public member may struggle to answer important operational and public safety questions such as:


    •       Which violation types are increasing over time? 

    •	Which neighborhoods show higher inspection risk? 

    •	Which establishments repeatedly fail inspections? 

    •	Which violations are most severe or most common? 

    •	What do specific violation codes and descriptions mean? 

    •	What corrective actions should businesses take to improve compliance? 


In addition, inspection datasets contain both structured numerical information and unstructured textual comments. Traditional dashboards and static reports are limited in their ability to interpret semantic patterns, summarize recurring issues, or explain contextual relationships between violations and inspection outcomes.


This project addresses these limitations by developing a domain-specific conversational AI analyst capable of processing Boston Food Establishment Inspection data using Retrieval-Augmented Generation (RAG), Large Language Models (LLMs), text-to-SQL analytics, and tool-calling frameworks. The proposed system allows users to ask natural-language questions and receive analytical responses, contextual summaries, data visualizations, and inspection insights without requiring technical expertise.


The system focuses on three primary capabilities:
1.	Contextual Understanding of Inspection Data
The model is enhanced with domain-specific knowledge of violation terminology, severity rankings, inspection standards, and food safety concepts to improve retrieval accuracy and analytical relevance. 


2.	Conversational Analytical Querying
The system combines deterministic SQL analytics with LLM-based reasoning to support flexible natural-language questions regarding trends, severity, repeated violations, and inspection performance. 


3.	Semantic Retrieval and Summarization
Through RAG and vector similarity search, the model analyzes inspection comments and violation descriptions to identify recurring patterns, summarize issues, and provide actionable insights for stakeholders.


## 2. Installation, Configuration and Set UP

### 2.1 Import

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import pyarrow as pa

In [ ]:
from dotenv import load_dotenv
# LangChain components for  RAG system
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
import duckdb
import subprocess
from pathlib import 
import faiss




/Users/jiezhao/miniconda3/envs/llm_langchain/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
import torch
#enable gpu 

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [3]:
# Load your environment variables
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("All libraries loaded successfully!")

All libraries loaded successfully!


## 3. Dataset and Data Preparation

Boston Food Establishment Inspections dataset contains the outcomes of food establishment inspections conducted in the city’s greater area since 2006. The dataset is open data source. Updated daily, this dataset provides details about individual inspections and results of businesses serving food. For this analysis, we are working with a static version of the dataset, comprising 27 columns, and 884608 individual records.

The dataset includes information such as the time and location of each inspection, the business entity responsible, and the licensing details. It also records inspection outcomes, violations noted, and any follow-up actions or comments, offering a comprehensive view of Boston’s food safety practices.

### 3.1 Load data and inspect

Boston food establish inspection dataset is downloaded from Boston goverment website 'https://data.boston.gov/dataset/food-establishment-inspections'. The dataset is stored in data folder. Two dataset is used in the project. 
    - food_inspection_report_raw.csv
    - A reference data file for results description. InspectionResult_description.csv

The public dataset documentation did not provide complete descriptions for all columns. Therefore, exploratory data analysis was performed to better understand the dataset structure, column meanings, data types, missing values, uniqueness, and overall data quality before implementing the analytical system.
 

#### 3.1.1. Load dataset using pandas.

In [4]:
#Load and inspect data
df = pd.read_csv('data/food_inspection_report_raw.csv')
print('structure of data:', df.info())


/var/folders/0x/_5wx0swx0vxgsyrkv847t75w0000gn/T/ipykernel_91445/1391034812.py:2: DtypeWarning: Columns (0: zip) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/food_inspection_report_raw.csv')


<class 'pandas.DataFrame'>
RangeIndex: 884608 entries, 0 to 884607
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   businessname  884608 non-null  str    
 1   dbaname       8431 non-null    str    
 2   legalowner    575201 non-null  str    
 3   namelast      884608 non-null  str    
 4   namefirst     508539 non-null  str    
 5   licenseno     884608 non-null  int64  
 6   issdttm       883712 non-null  str    
 7   expdttm       883928 non-null  str    
 8   licstatus     884608 non-null  str    
 9   licensecat    884608 non-null  str    
 10  descript      884608 non-null  str    
 11  result        884608 non-null  str    
 12  resultdttm    878210 non-null  str    
 13  violation     823807 non-null  str    
 14  viol_level    823807 non-null  str    
 15  violdesc      816937 non-null  str    
 16  violdttm      823804 non-null  str    
 17  viol_status   823807 non-null  str    
 18  status_date   3

#### 3.1.2 Inspect data structure, volume, columns.

In [5]:
# review data shape and data stats
print('='*100)
print('size of data', df.shape)
print('='*100)
print('columns', df.columns)
print('='*100)
print('describe of data', df.describe())


size of data (884608, 26)
columns Index(['businessname', 'dbaname', 'legalowner', 'namelast', 'namefirst',
       'licenseno', 'issdttm', 'expdttm', 'licstatus', 'licensecat',
       'descript', 'result', 'resultdttm', 'violation', 'viol_level',
       'violdesc', 'violdttm', 'viol_status', 'status_date', 'comments',
       'address', 'city', 'state', 'zip', 'property_id', 'location'],
      dtype='str')
describe of data            licenseno    property_id
count  884608.000000  727234.000000
mean   107065.636450  150178.259790
std    144712.285047  103322.491624
min        54.000000       0.000000
25%     22153.000000   77703.000000
50%     28531.000000  155991.000000
75%    125523.000000  157956.000000
max    624593.000000  460598.000000


In [6]:
# Look into a few row and inspect the data
df.head(5)

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,...,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
0,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,One staff person without hair restraint. Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,Caked on food debris on can opener blade. Clea...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,Menu was redesigned allergy statement was remo...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
3,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-08-08 15:54:00+00,Fail,NaN,Several dented cans found on storage shelves. ...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
4,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-08-08 15:54:00+00,Fail,NaN,Wet wiping cloths found on counter tops . Remo...,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"


### 3.2  Data columns inspection and transformation
The dataset columns were explored to identify variables relevant to food inspection analysis. This step involved examining each column’s data type, missing values, uniqueness, and cardinality to better understand the dataset structure and data quality.

Inspect violation and description columns: each violation description corresponds to a specific violation code. This helped validate the relationship between the coded inspection fields and the human-readable violation text used later in the analysis.


In [7]:
# inspect violation and description columns
print('violcation code carinality\n\n', df['violation'].value_counts())
print('='*100)
print('violcation description carinality\n\n',df['violdesc'].value_counts())

violcation code carinality

 violation
23-4-602.13            43973
37-6-501.11-.12        39951
15-4-202.16            35183
36-6-501.11-.12        33806
08-3-305-307.11        30211
                       ...  
590.004/4-204.123-C        1
02-3-305.11(2)             1
590.003/3-801.11-C         1
                           1
590.005/5-402.14-PF        1
Name: count, Length: 462, dtype: int64
violcation description carinality

 violdesc
Non-Food Contact Surfaces Clean                                                                   43973
Improper Maintenance of Walls/Ceilings                                                            39951
Non-Food Contact Surfaces                                                                         35183
Improper Maintenance of Floors                                                                    33806
Food Protection                                                                                   30211
                                      

Inspect violation level: column requires to map violation level from stars to understanable severity, and assign numerical scores to severity

In [8]:
# inspect violation level
df['viol_level'].value_counts()

viol_level
*       579434
***     126544
**      110958
-         6869
1919         1
             1
Name: count, dtype: int64

Inspect result column. We can group result into pass, failed, severed failed etc and assign risk scores.

In [9]:
#inspect result column
df['result'].value_counts()

result
HE_Fail       369707
HE_Pass       282095
HE_Filed       92851
HE_FailExt     73291
HE_Hearing     27393
HE_NotReq      23985
HE_TSOP         7669
HE_VolClos      2839
HE_OutBus       2521
Pass             964
HE_Closure       711
Fail             238
HE_FAILNOR       146
HE_Misc          128
DATAERR           42
HE_Hold           17
Failed             6
Closed             2
PassViol           2
NoViol             1
Name: count, dtype: int64

Inspect comment column, we can see the comments usually include the detailed observation of violation, and improvement suggestion.

In [10]:
pd.set_option('display.max_colwidth', None)
df['comments'][:8]

0                                                                               One staff person without hair restraint. Provide
1                                                                     Caked on food debris on can opener blade. Clean to remove.
2                      Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.
3    Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.
4                                      Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.
5                                                                                         No labels on bulk containers . Provide
6                                      Scoops found submerges in flour and other bulk container. Store properly with handles up.
7                                                                                                

In [21]:
#description of inspection result
ref = pd.read_csv('data/InspectionResult_description.csv')
# Standardize column names
ref.columns = ref.columns.str.lower().str.strip()
ref

,inspectionresult,inferreddescription
0,HE_Fail,Inspection failed; violations found
1,HE_Pass,Inspection passed; no issues found
2,HE_Filed,Minor violations found; no urgent follow-up required
3,HE_FailExt,Extended failure from prior inspection
4,HE_Hearing,Violations being addressed; follow-up required
5,HE_NotReq,Inspection not required; no immediate need
6,HE_TSOP,Temporary suspension of permit issued
7,HE_OutBus,Business closed; out of operation
8,HE_VolClos,Voluntary closure by business to avoid penalties
9,HE_Closure,Forced closure due to critical violations


Upon initial inspection of the data, we decided the columns below are relative and meaningful for our anlaysis.

- businessname: business name
- licenseno: Key identifier for the business/ restaurant
-result: result of  inspection
-resultdttm: date on which the results were generated
-violation: coding of law regulation related to violations
- viol_level: level of violation
-violdesc: reason of violation
- violdttm: date on which violation status was generated
- viol_status: status for violation: Fail or Pass
-status_date: date on which violation status was set to pass
- comments: comments given to the food establishment for improvement
- address: address of the business
- zipcode: zipcode of the business
- location: latitude and longitude of the business location

In [12]:
#extract column name, create describtion and decide action to keep or remove for the purpose of the analysis 
columns_name = df.columns
column_description = [
    "business name shown on inspection record.",
    "a different name of the business.",
    "owner of the business.",
    "owner's last name.",
    "owner's first name.",
    "business license number.",
    "license issue date/time.",
    "license expiration date/time.",
    "license status.",
    "license category code.",
    "business category.",
    "inspection result outcome.",
    "inspection result date/time.",
    "violation code.",
    "violation severity level/code.",
    "violation description.",
    "violation date/time.",
    "violation status.",
    "date violation status was updated.",
    "inspector comments or notes.",
    "business street address.",
    "business city.",
    "business state.",
    "business zip code.",
    "property identifier.",
    "geographic location coordinates."
]
column_action = [
    "keep", "remove", "remove", "remove", "remove",
    "keep", "remove", "remove", "remove", "remove",
    "remove", "keep", "keep", "keep", "keep",
    "keep", "keep", "keep", "remove", "keep",
    "keep", "keep", "remove", "keep", "remove", "keep"
]

column_review_df = pd.DataFrame({
    "column_name": columns_name,
    "column_description": column_description,
    "column_action": column_action
})

column_review_df.sort_values(by='column_action')

,column_name,column_description,column_action
0,businessname,business name shown on inspection record.,keep
5,licenseno,business license number.,keep
11,result,inspection result outcome.,keep
12,resultdttm,inspection result date/time.,keep
13,violation,violation code.,keep
14,viol_level,violation severity level/code.,keep
15,violdesc,violation description.,keep
16,violdttm,violation date/time.,keep
17,viol_status,violation status.,keep
19,comments,inspector comments or notes.,keep


In [18]:
cols = column_review_df[column_review_df['column_action'] == 'keep']['column_name']
data =  df[cols]

Tranformation

In [14]:
# mapping violation level from stars to understanable severity, and assign numerical scores to severity
severity_map = {
    "*": "low",
    "**": "medium",
    "***": "high",
    "-": None,
    "1919": None
}
severity_score_map= {
    "*": 1,
    "**": 2,
    "***": 3
}


We aggregate inspection results into category and assign risk scores

In [15]:
result_group_map = {
    "HE_Pass": "pass",
    "Pass": "pass",
    
    "HE_Filed": "minor_violation",

    "HE_Fail": "fail",
    "Fail": "fail",
    "Failed": "fail",
    "HE_FAILNOR": "fail",

    "HE_FailExt": "extended_fail",
    "HE_Hearing": "hearing",

    "HE_TSOP": "temporary_suspension",
    "HE_VolClos": "voluntary_closure_avoid",
    "HE_Closure": "forced_closure",

    "HE_OutBus": "out_of_business",
    "Closed": "closed",

    "HE_NotReq": "not_required",
    "HE_Misc": "misc",
    "DATAERR": "data_error",
    "HE_Hold": "hold"
}

risk_score_map = {
    "pass": 0,
    "minor_violation": 2,
    "hold": 2,

    "fail": 3,
    "extended_fail": 3,
    "hearing": 3,

    "temporary_suspension": 4,
    "voluntary_closure_avoid": 4,
    "forced_closure": 4,

    "out_of_business": None,
    "closed": None,
    "not_required": None,
    "misc": None,
    "data_error": None,
}



Map severity code to severity descritpion and map to severity score defined ealier. 
Map result code to risk descritpion and map to risck score defined ealier. 

In [19]:
data["severity_level"] = data["viol_level"].map(severity_map)
data["severity_score"] = data["viol_level"].map(severity_score_map)

data["result_group"] = data["result"].map(result_group_map)
data["risk_score"] = data["result_group"].map(risk_score_map)

In [22]:
# merge the description of inspection result to the main dataframe
data = data.merge(
    ref[["inspectionresult", "inferreddescription"]],
    left_on="result",
    right_on="inspectionresult",
    how="left"
)
data = data.drop(columns=["inspectionresult"])

In [23]:
# the analysis focuses on violation related content, we drop the rows with missing violation, result, viol_level, viol_status
data.dropna(subset=['violation','result','viol_level','viol_status'], inplace=True)


823,807 usable rows,1-2% percent of the data is missing.

In [25]:
# handling na
# need date time data for analysis 
data["resultdttm"] = pd.to_datetime(df["resultdttm"], errors="coerce")
data["resultdttm"] = pd.to_datetime(data["resultdttm"], errors="coerce")
data["year"] = data["resultdttm"].dt.year
data["month"] = data["resultdttm"].dt.strftime("%Y-%m")
data = data.dropna(subset=["resultdttm"])

#remove most current month 2026-05 as it's not completed month
data = data[data['month'] != '2026-05']
# fill na with unknown
data["violdesc"] = data["violdesc"].fillna("Unknown")
data["comments"] = data["comments"].fillna("No comment")
data["address"] = data["address"].fillna("Unknown")
data["inferreddescription"] = data["inferreddescription"].fillna("Unknown")
data["address"] = data["address"].fillna("Unknown")
data["zip"] = data["zip"].astype("string")


In [27]:
data.info()

<class 'pandas.DataFrame'>
Index: 819596 entries, 0 to 884607
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype              
---  ------               --------------   -----              
 0   businessname         819596 non-null  str                
 1   licenseno            819596 non-null  int64              
 2   result               819596 non-null  str                
 3   resultdttm           819596 non-null  datetime64[us, UTC]
 4   violation            819596 non-null  str                
 5   viol_level           819596 non-null  str                
 6   violdesc             819596 non-null  str                
 7   violdttm             819594 non-null  str                
 8   viol_status          819596 non-null  str                
 9   comments             819596 non-null  str                
 10  address              819596 non-null  str                
 11  city                 819596 non-null  str                
 12  zip               

In [90]:
df.head(3)

,businessname,dbaname,legalowner,namelast,namefirst,licenseno,issdttm,expdttm,licstatus,licensecat,...,violdttm,viol_status,status_date,comments,address,city,state,zip,property_id,location
0,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,One staff person without hair restraint. Provide,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
1,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,Caked on food debris on can opener blade. Clean to remove.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"
2,1000 Degrees Pizza,NaN,KHOSLA VIPAN,Pasquriello LLC,Kenneth Pasquariello,313440,2017-08-14 12:49:37+00,2020-01-01 04:59:00+00,Inactive,FS,...,2018-03-20 14:54:25+00,Fail,NaN,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,55 COURT ST,BOSTON,MA,02108,156226.0,"(42.35925954972639, -71.05890048027378)"


Write to parquet file for database use.

In [28]:
#write it to parquet file, more efficient for database use
data.to_parquet("data/food_inspections_clean.parquet", engine="pyarrow",index=False)

## 4. Step by step development

### 4.1 DuckDB setup

Load cleaned data into DuckDB

In [29]:
# connect to duckdb
con = duckdb.connect("data/food_inspections_clean.duckdb")
# create table
con.execute("""
CREATE OR REPLACE TABLE inspections AS
SELECT *
FROM read_parquet('data/food_inspections_clean.parquet')
""")

con.execute("SELECT COUNT(*) FROM inspections").fetchall()

[(819596,)]

### 4.2 Define SQL functions 

Aggregated data would usually surface interesting insights, we would like have understanding high level around questions below. 
Potential business question: what violation types are most common this year?”

In [111]:
# SQL functions 1 - top violation types,What violations are most common?
def top_violation_types(year=2025, severity_level='high', limit=10):
    # input: year, severity level, limit
    # output: dataframe of violation types and their counts
    # query: select violation type, count of violation
    query = """
    SELECT 
        violdesc,
        COUNT(*) AS violation_count
    FROM inspections
    WHERE violdesc IS NOT NULL 
      AND violdesc <> 'Unknown violation'
      AND result_group <> 'pass'
      AND severity_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
      AND (? IS NULL OR severity_level = ?)
    GROUP BY violdesc
    ORDER BY violation_count DESC
    LIMIT ?
    """
    return con.execute(
        query,
        [year, year, severity_level, severity_level, limit]
    ).df()

Business question: Are high-severity violations increasing faster than mild violations? Distinguishes whether risk is worsening or whether only minor violations are increasing.

In [85]:
# SQL function 2 - violation record trend by year and severity
def violation_count_by_year_and_severity(start_year=2016):
    query = """
    SELECT
        year,
        severity_level,
        COUNT(*) AS violation_record_count
    FROM inspections
    WHERE year >= ?
      AND result_group <> 'pass'
      AND severity_level IS NOT NULL
      AND violdesc IS NOT NULL
      AND violdesc <> 'Unknown violation'
    GROUP BY year, severity_level
    ORDER BY year, severity_level
    """
    return con.execute(query, [start_year]).df()

Business question: Which restaurants repeatedly fail inspections or have consistently severe violations?

For this function, we aim to understand owner violation seriousness. It can computer by severe risk score, serverity level or better, violation count, or better using combined results to generate an overall score. We define a serious violation score to measure how serious a violation pattern is overall for each business. the score combines both violation severity and inspection risk. The score is calculated as: bad_violation_score = AVG(severity_score) * 0.6 + AVG(risk_score) * 0.3+ LOG(COUNT(*) + 1) * 0.1 we give more weight to severity score because it directly reflects how serious the violation is, while risk score provides additional context about the inspection risk level.
We did also include count as factor too as we saw some owners have 32 violation ranks lower then the restrauant with only 4 violation. We use log transformations on counts as the count have bigger range,  for skewed count variables they reduce the influence of very large values while preserving the signal from frequency.

In [47]:
#SQL functions 3 - top violation owner
def top_violation_owner(year=2025, limit=15, rank_by="score"):
    # input: year, limit, rank_by
    # output: dataframe of business name, address, total violations, high severity violation, high risk violation, average severity score, average risk score, average seriousness score
    # query: select business name, address, count of violation, high severity violation, high risk violation, average severity score, average risk score, average seriousness score 
    # serious level of violation is computed using both severity score and risk score
    query = """
    SELECT
        businessname,
        address,
        COUNT(*) AS total_violations,
        SUM(CASE WHEN severity_score >= 3 THEN 1 ELSE 0 END) AS high_severity_count, 
        SUM(CASE WHEN risk_score >= 3 THEN 1 ELSE 0 END) AS high_risk_count, 
        ROUND(AVG(severity_score), 2) AS avg_severity_score,
        ROUND(AVG(risk_score), 2) AS avg_risk_score,
        ROUND(AVG(severity_score) * 0.6 + AVG(risk_score) * 0.3+ LOG(COUNT(*) + 1) * 0.1,2) as overall_violation_score
    FROM inspections
    WHERE businessname IS NOT NULL
      AND severity_score IS NOT NULL
      AND risk_score IS NOT NULL
      AND (? IS NULL OR EXTRACT(year FROM resultdttm) = ?)
    GROUP BY businessname, address
    HAVING COUNT(*) >= 3
    ORDER BY
        CASE
            WHEN ? = 'score' THEN overall_violation_score
            WHEN ? = 'high_count' THEN high_severity_count
            WHEN ? = 'risk_count' THEN high_risk_count
            WHEN ? = 'total_count' THEN total_violations
            ELSE overall_violation_score
        END DESC
    LIMIT ?
    """

    return con.execute(
        query,
        [year, year, rank_by, rank_by, rank_by, rank_by, limit]
    ).df()

Test SQL results

In [40]:
# sql 1 test
top_violation_types(2025)

,violdesc,violation_count
0,(A)(2) and (B) Time/Temperature Control for Safety Food Hot and Cold Holding (P),1051
1,Packaged and Unpackaged Food-Separation Packaging and Segregation (P),550
2,(A)(1) Time/Temperature Control for Safety Food Hot and Cold Holding (P),427
3,Manual and Mechanical Warewashing Equipment Chemical Sanitization-Temperature pH Concentration and Hardness (P),273
4,Backflow Prevention (P),237
5,Backflow Prevention Air Gap (P),200
6,System Maintained in Good Repair (P),189
7,Discarding or Reconditioning Unsafe Adulterated or Contaminated Food (P),149
8,When to Wash (P),139
9,Cooling (P),134


In [48]:
top_violation_owner(2024,10)

,businessname,address,total_violations,high_severity_count,high_risk_count,avg_severity_score,avg_risk_score,overall_violation_score
0,Ogawa Coffee,10 MILK ST,5,5.0,3.0,3.00,2.2,2.54
1,El Pupi Chimi,1 CITYWIDE ST,4,1.0,4.0,2.00,4.0,2.47
2,Freeport Street Cafeteria,179 FREEPORT ST,3,3.0,2.0,3.00,2.0,2.46
3,Ethiopian Cafe,377 CENTRE ST,32,11.0,32.0,2.22,3.0,2.38
4,Boston Halal,273 HUNTINGTON AV,3,2.0,3.0,2.33,3.0,2.36
5,Beantown Kebab,44 KILBY ST,3,2.0,3.0,2.33,3.0,2.36
6,Bread Thyme,1868 CENTRE ST,5,3.0,4.0,2.60,2.4,2.36
7,Pho Zabb,1799 COMMONWEALTH AV,6,6.0,3.0,3.00,1.5,2.33
8,Camilo's Market II,204 WASHINGTON ST,4,4.0,2.0,3.00,1.5,2.32
9,FMI Cafe - 6th Fl. Main Kitchen,370 SUMMER ST,4,4.0,2.0,3.00,1.5,2.32


4.2 Define Chart Function

Test chart function

In [86]:
def chart_violation_count_by_year_and_severity(df):
    fig = px.line(
        df,
        x="year",
        y="violation_record_count",
        color="severity_level",
        markers=True,
        title="Violation Record Count by Year and Severity"
    )

    fig.update_layout(
        xaxis_title="Year",
        yaxis_title="Violation Record Count",
        legend_title_text="Severity Level"
    )

    return fig

In [87]:
df_trend = violation_count_by_year_and_severity(2016)
fig = chart_violation_count_by_year_and_severity(df_trend)
fig.show()

In [ ]:
def chart_top_violation_types(df):
    fig = px.bar(
        df,
        x="violation_count",
        y="violdesc",
        orientation="h",
        title="Top Violation Types",
        text="violation_count"
    )

    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        xaxis_title="Violation Count",
        yaxis_title="Violation Type"
    )

    return fig

In [204]:
df_top_types = top_violation_types(2025)
chart_top_violation_types(df_top_types)

In [222]:
#chart to show total violation records for each business
def chart_top_violation_owner(df, year=None):
    df = df.copy()

    title_year = f" in {year}" if year else " Overall"

    fig = px.bar(
        df.sort_values("total_violations"),
        x="total_violations",
        y="businessname",
        orientation="h",
        title=f"Top Businesses by Total Violation Records{title_year}",
        hover_data={
            "high_severity_count": True,
            "high_risk_count": True,
            "avg_severity_score": ":.2f",
            "avg_risk_score": ":.2f",
            "overall_violation_score": ":.2f",
            "businessname": False
        }
    )

    fig.update_layout(
        template="plotly_white",
        height=500,
        title_x=0.5,
        xaxis_title="Total Violation Records",
        yaxis_title="Business"
    )

    return fig

In [208]:
df_owner = top_violation_owner(year=2024, limit=15)
chart_top_violation_owner(df_owner, year=2024)

In [212]:
def chart_owner_risk_scatter(df, year=None):
    df = df.copy()

    title_year = f" in {year}" if year else " Overall"

    fig = px.scatter(
        df,
        x="total_violations",
        y="overall_violation_score",
        size="high_risk_count",
        color="avg_severity_score",
        hover_name="businessname",
        hover_data={
            "high_severity_count": True,
            "high_risk_count": True,
            "avg_risk_score": ":.2f",
            "avg_severity_score": ":.2f",
            "overall_violation_score": ":.2f"
        },
        title=f"Business Violation Volume vs Overall Violation Score{title_year}"
    )

    fig.update_layout(
        template="plotly_white",
        height=600,
        title_x=0.5,
        xaxis_title="Total Violation Records",
        yaxis_title="Overall Violation Score"
    )

    return fig

In [218]:
df_owner = top_violation_owner(year=2025, limit=20, rank_by="score")
fig = chart_owner_risk_scatter(df_owner, year=2025)
fig.show()

### 4.5. Deep learning / embeddings

DataFrame row
→ formatted text document
→ embedding vector
→ FAISS index
→ retrieve similar rows
→ LLM summary

#### 4.5.1 Prepare document



From the closer look at the sample: the same violation comments can occur in both Pass and Fail records, so severity depends on broader inspection context, not only the text. It indicates the need for using both structured fields plus comments together.

In [ ]:
data[["comments", "risk_score", "severity_score", "violation",'result','viol_status']].head(20)

,comments,risk_score,severity_score,violation,result,viol_status
0,One staff person without hair restraint. Provide,3.0,1.0,13-2-304/402.11,HE_Fail,Fail
1,Caked on food debris on can opener blade. Clean to remove.,3.0,2.0,22-4-601/602.11,HE_Fail,Fail
2,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,3.0,3.0,M-2-103.11,HE_Fail,Fail
3,Several dented cans found on storage shelves. Remove . Set up area for returns PIC removed and is returning to distributor.,2.0,1.0,08-3-305-307.11,HE_Filed,Fail
4,Wet wiping cloths found on counter tops . Remove . Store properly in saniitizer when wet.,2.0,1.0,21-3-304.14,HE_Filed,Fail
5,No labels on bulk containers . Provide,2.0,1.0,02-3-602.11-.12/3-302.12,HE_Filed,Fail
6,Scoops found submerges in flour and other bulk container. Store properly with handles up.,2.0,1.0,10-3-304.12,HE_Filed,Fail
10,One staff person without hair restraint. Provide,1.0,1.0,13-2-304/402.11,HE_Pass,Pass
11,Caked on food debris on can opener blade. Clean to remove.,1.0,2.0,22-4-601/602.11,HE_Pass,Pass
12,Menu was redesigned allergy statement was removed. Provide proper allery statement for customers to read.,1.0,3.0,M-2-103.11,HE_Pass,Pass


Prepare documentation for RAG by combining meanningful fields into one text block for each records.

In [92]:
#prepare document for each row, only extract meaningful fields 
def build_rag_text(row):
    fields = {
        "business": row.get("businessname"),
        "address": row.get("address"),
        "zip": row.get("zip"),
        "violation": row.get("violdesc"),
        "severity": row.get("severity_level"),
        "inspection result": row.get("result_group"),
        "inspection result code": row.get("result"),
        "inspection result meaning": row.get("inferreddescription"),
        "violation_status": row.get("viol_status"),
        "Inspector Observation / Corrective Action": row.get("comments")
    }
    # join the fields into a single text
    parts = []
    for label, value in fields.items():
        if pd.notna(value) and str(value).strip() != "":
            parts.append(f"{label}: {value}")

    return "\n".join(parts)

In [93]:
# Create rag dataframe from cleaned data
rag_df = data.copy()
# build rag text
rag_df["rag_text"] = rag_df.apply(build_rag_text, axis=1)

In [99]:
#inspect the first string in the rag text field
print(rag_df["rag_text"].iloc[0])

business: 1000 Degrees Pizza
address: 55  COURT ST
zip: 02108
violation: Clean Cloths  Hair Restraint
severity: low
inspection result: fail
inspection result code: HE_Fail
inspection result meaning: Inspection failed; violations found
violation_status: Fail
Inspector Observation / Corrective Action: One staff person without hair restraint. Provide


#### 4.5.2 Generate embedding

consideration: choose the embedding model that balance cost and efficiency and performance
Experiment with lightweight model all-MiniLM-L6-v2, fast, resource-efficient embeddings.  With just 22M parameters, it delivers solid performance on general semantic search tasks and is used across many production-grade apps.



In [100]:
#combine all text into a list
texts = rag_df["rag_text"].tolist()
#initiate embedding model
embed_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)
# apply embedding to the text
embeddings = embed_model.encode(
    texts,
    batch_size=128 if device == "mps" else 64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/6404 [00:00<?, ?it/s]

In [101]:
np.save("data/rag_embeddings.npy", embeddings)
rag_df.to_parquet("data/rag_df.parquet")

Build FAISS index

In [ ]:

embeddings = embeddings.astype("float32")
print("embeddings shape", embeddings.shape)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS index size:", index.ntotal)
# write index to file
faiss.write_index(index, "data/faiss_index.index")

embeddings shape (819596, 384)
FAISS index size: 819596


 RAG retrieval

In [105]:
def retrieve_similar_violations(query, k=5):
    #convert query to same embedding
    query_embedding = embed_model.encode(
        #embeds query to a list as faiss expect 2D array
        [query],
        normalize_embeddings=True
    ).astype("float32")
    #search for top k similar violations and store scores and indices
    scores, indices = index.search(query_embedding, k)
    #get the results
    results = rag_df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]

    return results[
        [
            "similarity_score",
            "businessname",
            "address",
            "violdesc",
            "severity_level",
            "result_group",
            "result",
            "inferreddescription",
            "comments"
        ]
    ]

The step test whether the retrieval system returns relevant inspection records for a sample user question. 

In [114]:
query = "Find similar violation patterns related to food spoilation"

retrieved_df = retrieve_similar_violations(query,5)
retrieved_df

,similarity_score,businessname,address,violdesc,severity_level,result_group,result,inferreddescription,comments
350337,0.608371,Halal Indian Cuisine,736 HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,fail,HE_Fail,Inspection failed; violations found,Interior of house model refrigerator unitv observed soiled with dry food spills upright freezer unit observed with heavy ice build up cook range observed heavily soiled with dry food encrusted heavy carbon build up and food spills interior of oven observed soiled interior of middle compartment of 3 bay sink observed soiled by dishes and foods front of the house soda refrigerator shelvings observed soiled wall behind cookline observed heavily soiled by dry food spills multiple kitchen shelvings observed soiled exterior of microwave all condiment holders observed soiled . Clean and sanitize all the above equipment and surfaces.
108875,0.604384,BRIGHAM CIRCLE CHINESE FOOD,728 HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,fail,HE_Fail,Inspection failed; violations found,Interior and exterior of multiple refrigeration units (shelving/ gastkets/floor) observed soiled exterior of numerous storage containers observed heavily soiled with dry foods encrust and visible food spills cook range observed heavily soiled chest refrigerator unit observed with heavy ice build up and food spills interior and exterior of microwave exterior of multiple rice cooker exterior of squeeze bottles and condiment containers observed soiled (clean and relabel ) and in between equipments observed with dry food encrusted. Clean and sanitize to remove.
509853,0.601675,Montecristo Mexican Grille,748A HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,fail,HE_Fail,Inspection failed; violations found,Interior of 4 chest freezer observed without heavy ice build up interior of 2 door refrigerator unit observed soiled with visible food spills exterior of bulk bins observed soiled by visible food spills and crumbs interior of beverage unit observed soiled shelvings inside walkin unit observed soiled and rusty interior of middle compartment of 3 bay sink observed soiled from foods exterior of multiple foods contact surfaces observed soiled by visible food spills.. Clean and sanitize to remove
509903,0.595236,Montecristo Mexican Grille,748A HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,extended_fail,HE_FailExt,Extended failure from prior inspection,Interior of 4 chest freezer observed without heavy ice build up interior of 2 door refrigerator unit observed soiled with visible food spills exterior of bulk bins observed soiled by visible food spills and crumbs interior of beverage unit observed soiled shelvings inside walkin unit observed soiled and rusty interior of middle compartment of 3 bay sink observed soiled from foods exterior of multiple foods contact surfaces observed soiled by visible food spills.. Clean and sanitize to remove
109145,0.587940,BRIGHAM CIRCLE CHINESE FOOD,728 HUNTINGTON AV,(A) Equipment Food-Contact Surfaces Nonfood-Contact Surfaces and Utensils (Pf),medium,pass,HE_Pass,Inspection passed; no issues found,Interior and exterior of multiple refrigeration units (shelving/ gastkets/floor) observed soiled exterior of numerous storage containers observed heavily soiled with dry foods encrust and visible food spills cook range observed heavily soiled chest refrigerator unit observed with heavy ice build up and food spills interior and exterior of microwave exterior of multiple rice cooker exterior of squeeze bottles and condiment containers observed soiled (clean and relabel ) and in between equipments observed with dry food encrusted. Clean and sanitize to remove.


format retrieve result into context block

In [115]:
def format_rag_context(results_df, max_rows=5):
    rows = []

    for i, row in results_df.head(max_rows).iterrows():
        rows.append(f"""
            Record {i+1}
            Business: {row.get("businessname")}
            Violation: {row.get("violdesc")}
            Severity: {row.get("severity_level")}
            Result: {row.get("result_group")}
            Inspection Result: {row.get("inferreddescription")}
            Comment: {row.get("comments")}
            """.strip())

    return "\n\n".join(rows)

In [116]:
format_rag_context(retrieved_df)

'Record 350338\n            Business: Halal Indian Cuisine\n            Violation: (A) Equipment  Food-Contact Surfaces  Nonfood-Contact Surfaces  and Utensils (Pf)\n            Severity: medium\n            Result: fail\n            Inspection Result: Inspection failed; violations found\n            Comment: Interior of house model refrigerator unitv observed soiled with dry food spills  upright freezer unit observed with heavy ice build up  cook range observed heavily soiled with dry food encrusted heavy carbon build up  and food spills  interior of oven observed soiled  interior of middle compartment of 3 bay sink observed soiled by dishes and foods  front of the house soda refrigerator shelvings observed soiled  wall behind cookline observed heavily soiled by dry food spills  multiple kitchen shelvings observed soiled  exterior of microwave  all condiment holders observed soiled . Clean and sanitize all the above equipment and surfaces.\n\nRecord 108876\n            Business: BRIGH

Use langhchain template to build summary prompt. good prompt will use two or more of them: 
* Instructions
* External information or context
* User input or query
* Output indicator

In [119]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2 #lower temperature means the model is less random and more consistent.most deterministic, better for routing SQL tool selection.
)

1. Use prompt template

In [120]:
rag_template = """

You are a food safety intelligence assistant.

User Question:
{question}

Retrieved Inspection Records:
{context}

Tasks:
1. Summarize the common food safety pattern.
2. Explain likely operational causes.
3. Suggest practical corrective actions restaurants should take.
4. Do not invent facts outside the retrieved records.

Answer:
"""

rag_prompt = PromptTemplate(
    template=rag_template,
    input_variables=["question", "context"]
)

rag_chain = rag_prompt | llm | StrOutputParser()

Invoke chain

In [121]:
query = "Find violations similar to sanitizer or dish machine problems."

retrieved_df = retrieve_similar_violations(query, 5)

context = format_rag_context(retrieved_df)

response = rag_chain.invoke({
    "question": query,
    "context": context
})

print(response)

### 1. Common Food Safety Pattern
The inspection records indicate a recurring issue with improper sanitization practices in restaurants. Specifically, there are multiple instances of low-temperature dish machines failing to test for sanitizer, as seen in the records from CAFE MIRROR and the Vintage Lounge. Additionally, there are critical violations related to the lack of proper sanitization methods, such as the absence of sanitizer in the dishwashing process and improper handling of chemicals. The records also highlight a lack of knowledge among staff regarding safe food handling practices, which contributes to unsanitary conditions.

### 2. Likely Operational Causes
The operational causes behind these violations may include:
- **Inadequate Training**: Staff may not be properly trained in food safety principles, leading to improper handling of food and equipment.
- **Equipment Malfunction**: Dish machines may not be functioning correctly, leading to insufficient sanitization of utensi

2.Build Rag llm anwser template, Use chat prompt template with defined syster, human,

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
            You are a food safety inspection analyst.
            Use only the retrieved inspection records.
            Do not add causes, recommendations, or facts unless supported.
            If the records do not provide enough evidence, say so.
            Keep the answer concise and evidence-based.
            """),
    ("human", """
            User question:
            {question}

            Retrieved inspection records:
            {context}

            Answer using this format:

            1. Common pattern:
            2. Evidence from retrieved records:
            3. Practical corrective actions supported by the records:

""")
])

rag_chain = rag_prompt | llm | StrOutputParser()

In [143]:
query = "What recurring corrective actions appear in Boston inspection comments related to low-temperature dish machines?"

retrieved_df = retrieve_similar_violations(query, 5)
context = format_rag_context(retrieved_df)

response = rag_chain.invoke({
    "question": query,
    "context": context
})

print(response)

1. Common pattern: Recurring issues with low-temperature dish machines and high-temperature dish machines not reaching required sanitization temperatures, leading to service calls for repairs and temporary use of alternative washing methods.

2. Evidence from retrieved records: 
   - Record 776047: The bar low temp glass machine's sanitizer was not registering, leading to it being taken out of service.
   - Record 94367 and 94351: The high-temperature warewashing machine in the barrel room did not exceed 149F after multiple attempts, prompting a service call for repairs and the use of a different warewashing machine.

3. Practical corrective actions supported by the records: 
   - Taking malfunctioning machines out of service until properly repaired.
   - Utilizing alternative warewashing methods (e.g., 3 bay sink or different location's warewashing machine) during repairs.


In [148]:
retrieved_df[
    [
        "similarity_score",
        "businessname",
        "violdesc",
        "severity_level",
        "comments"
    ]
].head(5)

,similarity_score,businessname,violdesc,severity_level,comments
776046,0.706270,The Bostonian Boston - A Millennium Hotel,Food Contact Surfaces Clean,high,High temperature main dish machine - 140F wash 140F rinse new heating element on order. Machine still in use during inspection / Machine will now be taken out of service until properly repaired / 3 bay sink will be used in the meantime Bar low temp glass machine - Sanitizer not registering / Machine taken out of service until properly repaired 3 bay sink will be used.
776321,0.703605,The Bostonian Boston - A Millennium Hotel,Food Contact Surfaces Clean,high,High temperature main dish machine - 140F wash 140F rinse new heating element on order. Machine still in use during inspection / Machine will now be taken out of service until properly repaired / 3 bay sink will be used in the meantime Bar low temp glass machine - Sanitizer not registering / Machine taken out of service until properly repaired 3 bay sink will be used.
101864,0.700029,Boston Park Plaza (Main Kitchen/Room Serv.),(A) and (C) Good Repair and Calibration-Utensils and Temperature and Pressure Measuring Devices (C),low,A service call was conducted on 01/19/23 for repairs to be made to the high temperature machine on the main level. Following that service it was determined that multiple parts needed to be replaced and parts are on order. Email confirmation of service completed on 01/19/23 Facility has large high temperature dish machne in the kitchen on the second level and all warewashing is being done in that machine until proper repairs are made to the machine in the main kitchen.
94366,0.676362,Boston Chops Downtown,Mechanical Warewashing Equipment Hot Water Sanitization Temperatures (Pf),medium,At the high temperature warewashing machine in the barrel room the temperature at the plate using the location's irreversible thermometer did not go above 149F after four attempts. The person in charge placed a call to get the unit repaired and the staff in the barrel room will utilize a different location's warewashing machine. All other high temperature dish machines were observed working properly.
94350,0.675533,Boston Chops Downtown,Mechanical Warewashing Equipment Hot Water Sanitization Temperatures (Pf),medium,At the high temperature warewashing machine in the barrel room the temperature at the plate using the location's irreversible thermometer did not go above 149F after four attempts. The person in charge placed a call to get the unit repaired and the staff in the barrel room will utilize a different location's warewashing machine. All other high temperature dish machines were observed working properly.


adding instruction: Evidence from retrieved records and Practical corrective actions supported by the records produce more ground base anwsers compared to previous promt usage.


Baseline pure LLM model result

In [144]:
# Plain LLM baseline: no SQL, no RAG, no retrieved records

plain_llm_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a food safety inspection analyst.
Answer the user question based on the provided document.
Keep the answer concise.
"""),
    ("human", """
User question:
{question}
""")
])

plain_llm_chain = plain_llm_prompt | llm | StrOutputParser()

In [145]:
raw_sample = rag_df.sample(20, random_state=42)["rag_text"].to_string(index=False)

plain_response = plain_llm_chain.invoke({
    "question": query,
    "raw_sample": raw_sample
})
print(plain_response)

Recurring corrective actions in Boston inspection comments related to low-temperature dish machines typically include ensuring proper chemical concentrations, verifying that the machines are functioning correctly, and conducting regular maintenance checks to prevent malfunctions.


In [146]:
plain_llm_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a food safety inspection analyst.
Answer the user questions based on your general knowledge.
Keep the answer concise.
"""),
    ("human", """
User question:
{question}
""")
])

plain_llm_chain = plain_llm_prompt | llm | StrOutputParser()

In [147]:


plain_response = plain_llm_chain.invoke({
    "question": query,
})
print(plain_response)

Recurring corrective actions for low-temperature dish machines in Boston inspection comments typically include:

1. **Chemical Concentration Adjustment**: Ensuring the proper concentration of sanitizing chemicals.
2. **Temperature Monitoring**: Regular checks to confirm that the machine is operating within the required temperature range.
3. **Maintenance and Repairs**: Addressing mechanical issues that affect performance, such as leaks or malfunctioning parts.
4. **Staff Training**: Providing training for staff on proper operation and monitoring of the dish machine.
5. **Cleaning and Maintenance**: Regular cleaning of the machine to prevent buildup that can affect sanitization.

These actions aim to ensure that the dish machines effectively sanitize dishes and utensils.


In [138]:
keyword_sample = rag_df[
    rag_df["rag_text"].str.contains(
        "sanitizer|dish|dishwasher|dish machine|machine",
        case=False,
        na=False
    )
]["rag_text"].head(10).to_string(index=False)

keyword_response = plain_llm_chain.invoke({
    "question": question,
    "raw_sample": keyword_sample
})
print(keyword_response)

Common violations related to sanitizer or dish machine issues in Boston restaurants include:

1. **Improper Sanitizer Concentration**: Sanitizer solutions not at the correct concentration for effective disinfection.
2. **Inadequate Dish Machine Temperature**: Dish machines failing to reach required temperatures for sanitizing dishes.
3. **Lack of Test Strips**: Absence of test strips to verify sanitizer concentration.
4. **Dirty or Clogged Spray Arms**: Dish machines with obstructed spray arms affecting cleaning efficacy.
5. **Improper Use of Sanitizers**: Using non-approved sanitizers or incorrect application methods.

For specific incidents, check local health department inspection reports.


Based on the comparison above performed on question, Plain LLM: gives generic best-practice actions.
RAG + LLM: gives actions grounded in actual Boston records with sepecific anwsers eg, sepecific desired temperature. It captures specific records
: specific machine conditions, sanitizer not registering, machine taken out of service


2.Build sql llm anwser template

In [149]:
sql_summary_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a food safety data analyst.

Use only the provided SQL result.
Do not invent numbers,names, years, causes, or trends.
If the SQL result is limited, say so.
Keep the answer concise.
Use 2–4 short bullet points maximum.
Do not restate every monthly value unless specifically requested.
"""),
    ("human", """
        User question:
        {question}

        SQL tool used:
        {tool_name}

        SQL result:
        {result}

Write the answer using this format:

1. Direct answer:
2. Key evidence from SQL result:
3. Business interpretation:
4. Limitation:
""")
])

sql_summary_chain = sql_summary_prompt | llm | StrOutputParser()

In [152]:
#create a wrapper function that runs a SQL tool, converts the dataframe to text
def sql_summary(question, tool_name, result):
    summary = sql_summary_chain.invoke({
    "question": question,
    "tool_name": tool_name,
    "result": result
})
    return summary


In [155]:
question_sql1 = "Show violation record count trends by year and severity since 2016."

df_result = violation_count_by_year_and_severity(start_year=2016)

summary = sql_summary(
    question=question_sql1,
    tool_name="violation_count_by_year_and_severity",
    result=df_result
)

print(summary)

1. Direct answer: Violation record counts show varying trends by year and severity from 2016 to 2026.

2. Key evidence from SQL result:
   - High severity violations peaked in 2016 (6758) and decreased significantly by 2020 (1820).
   - Low severity violations decreased from 28597 in 2016 to 13120 in 2020, then fluctuated slightly in subsequent years.
   - Medium severity violations showed a peak in 2021 (7335) after a decline in 2020.

3. Business interpretation: The data indicates a general decline in high and low severity violations over the years, suggesting improved compliance or enforcement. However, medium severity violations have shown variability, indicating potential areas for targeted interventions.

4. Limitation: The SQL result does not provide complete data for all years up to 2023, and future projections beyond 2026 may not reflect actual trends.


Inspect retrieved data result from sql

In [172]:
df_result[df_result["severity_level"] == "high"]

,year,severity_level,violation_record_count
0,2016.0,high,6758
3,2017.0,high,4991
6,2018.0,high,6937
9,2019.0,high,4022
12,2020.0,high,1820
15,2021.0,high,2951
18,2022.0,high,3213
21,2023.0,high,2520
24,2024.0,high,3233
27,2025.0,high,3088


### benchmark: sql-to-text with llm-generated sql

To evaluate the sql-to-text workflow, test whether the llm could translate natural language questions into executable duckdb sql queries and then summarize the query results in plain language.

In [ ]:
text_to_sql_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a DuckDB SQL analyst.

Table: inspections

Important columns:
- year: inspection year
- severity_level: human-readable severity label
- violdesc: violation description
- result_group: normalized inspection result group
- commetns 

Rules:
- Generate SELECT SQL only.
- Filter out unknown or missing violation descriptions.
- Do not include result_group = 'pass' for violation trend questions.
"""),
    ("human", """
User question:
{question}
""")
])

text_to_sql_chain = text_to_sql_prompt | llm | StrOutputParser()

In [165]:
text_to_sql_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a DuckDB SQL analyst.

Table: inspections

Schema:
- businessname: restaurant/business name
- address: business address
- zip: ZIP code
- resultdttm: inspection/result datetime
- year: extracted inspection year
- month: extracted inspection month
- violdesc: violation description
- severity_level: human-readable severity label
- severity_score: numeric severity score; higher = more severe
- result: raw inspection result code
- result_group: normalized result category
- risk_score: numeric inspection risk score; higher = higher risk
- comments: inspector comments / corrective action notes

Rules:
- Generate SELECT SQL only.
- Return only raw SQL text.
- Filter out unknown or missing violation descriptions.
- Do not include result_group = 'pass' for violation trend questions.
-Do not include ```sql or ```.
"""),
    ("human", """
User question:
{question}
""")
])

text_to_sql_chain = text_to_sql_prompt | llm | StrOutputParser()

In [227]:
chart_code_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a Python Plotly chart creator.

Given a pandas dataframe schema and user question, generate Python code to create a Plotly chart.

Rules:
- Use plotly.express as px.
- Assume dataframe is named df.
- Return only Python code.
- Do not include markdown.
- Do not reinvent data that doesn't exist.
- Do not modify df.
"""),
    ("human", """
User question:
{question}

Dataframe columns:
{columns}
""")
])

chart_code_chain = chart_code_prompt | llm | StrOutputParser()

In [ ]:
generated_sql = text_to_sql_chain.invoke({
    "question": question_sql1
})

print(generated_sql)

SELECT year, severity_level, COUNT(*) AS violation_count
FROM inspections
WHERE year >= 2016 AND violdesc IS NOT NULL
GROUP BY year, severity_level
ORDER BY year, severity_level;


In [ ]:
text_to_sql_df = con.execute(generated_sql).df()
text_to_sql_summary = sql_summary(
    question=question_sql1,
    tool_name="text_to_sql_generated_query",
    result=text_to_sql_df
)

print(text_to_sql_summary)

1. Direct answer: The SQL result does not provide specific businesses with violations for 2025.
2. Key evidence from SQL result: The data shows violation counts for different severity levels in 2025, but lacks business identification.
3. Business interpretation: Without business-specific data, it's impossible to identify which businesses are most problematic in terms of violations.
4. Limitation: The SQL result is limited as it does not include business names or identifiers related to the violations.


In [229]:
question2 = "What are the top 10 businesses with the most violations in 2025? Show a chart."

generated_sql2 = text_to_sql_chain.invoke({"question": question2})
df2 = con.execute(generated_sql2).df()

chart_code = chart_code_chain.invoke({
    "question": question2,
    "columns": list(df2.columns)
})

print(chart_code)

import plotly.express as px

top_violations = df.nlargest(10, 'violation_count')
fig = px.bar(top_violations, x='businessname', y='violation_count', title='Top 10 Businesses with Most Violations in 2025')
fig.show()


The predefined SQL function and the LLM-generated text-to-SQL query produced different analytical results for the same business question, example: regarding yearly high-severity violation trends:

The predefined SQL function reported that high-severity violations peaked in 2016 with 6,758 records and declined to 1,820 records by 2020. In contrast, the LLM-generated text-to-SQL approach reported a higher peak in 2018 with 10,235 records.

Even we defined the business rules do not include pass in violation record, the generated sql still includes those record. Prompt instructions alone do not guarantee enforcement of business logic. Possible reason includes:
- prompts are long
- multiple rules exist
- the question itself does not explicitly mention the rule
- the model optimizes for “plausible SQL”

In [ ]:
benchmark_df = pd.DataFrame({
    "approach": [
        "Text-to-SQL",
        "Predefined SQL function"
    ],
    "reliability": [
        "Flexible but may generate inconsistent or incorrect SQL",
        "Deterministic and repeatable"
    ],
    "uses_business_rules": [
        "Define schema and business rules clearly",
        "Not all business rulse are used as prompt"
    ],
    "result_rows": [
        len(text_to_sql_df),
        len(rag_df)
    ],
    "summary": [
        text_to_sql_summary,
        summary
    ]
})

benchmark_df

,approach,reliability,uses_business_rules,result_rows,summary
0,Text-to-SQL,Flexible but may generate inconsistent or incorrect SQL,Only if prompt/schema describes them clearly,41,"1. Direct answer: Violation record counts show varying trends by year and severity from 2016 to 2026.\n\n2. Key evidence from SQL result:\n - High severity violations peaked in 2018 (10,235) and have generally declined since.\n - Low severity violations decreased from 39,892 in 2016 to 21,878 in 2023.\n - Medium severity violations fluctuated, with a high of 11,011 in 2021.\n\n3. Business interpretation: There is a noticeable decline in high and low severity violations over the years, indicating potential improvements in food safety practices. However, medium severity violations remain significant and require attention.\n\n4. Limitation: The SQL result does not provide complete data for all years, particularly for 2024 to 2026, which may affect trend analysis."
1,Predefined SQL function,Deterministic and repeatable,"Yes, encoded directly in function",819596,"1. Direct answer: Violation record counts show varying trends by year and severity from 2016 to 2026.\n\n2. Key evidence from SQL result:\n - High severity violations peaked in 2016 (6758) and decreased significantly by 2020 (1820).\n - Low severity violations decreased from 28597 in 2016 to 13120 in 2020, then fluctuated slightly in subsequent years.\n - Medium severity violations showed a peak in 2021 (7335) after a decline in 2020.\n\n3. Business interpretation: The data indicates a general decline in high and low severity violations over the years, suggesting improved compliance or enforcement. However, medium severity violations have shown variability, indicating potential areas for targeted interventions.\n\n4. Limitation: The SQL result does not provide complete data for all years up to 2023, and future projections beyond 2026 may not reflect actual trends."


RAG summary chain
SQL summary chain
Router chain
Final ask() demo function

bench mark vs llm without rag, sql, chart.
Good comparison questions

Use 3–5 examples:

“What are the most common Boston food inspection violations in 2024?”
“Which establishments had the most serious outcomes in 2024?”
“Find violations similar to refrigeration or cold food storage problems.”
“What corrective actions are common for sanitizer machine issues?”
“Are high-severity violations increasing?”
Evaluation table
Question	LLM-only result	RAG/SQL result	Which is better?	Why

In [84]:
test_question = ['What are most common violantions in 2025',
                'Find pest or rodent activity violations.',
                'what are common pattern for pest or rodent activity violations?',
                'Based on similar cases, what actions should a restaurant take to fix sanitizer violations?
                'Are these refrigeration-related violations usually serious?'
                'Find violations similar to sanitizer or dish machine problems.',
                'Should a customer be concerned about repeated cold holding violations?',
                'What are high severity violations in 2024?',
                'Find violations similar to refrigeration or cold food storage problems.',
                'What corrective actions are common for sanitizer machine issues?',
                'Are high-severity violations increasing?',
                
                    ]

SyntaxError: unterminated string literal (detected at line 4) (504954163.py, line 4)

8. Router

Router template: when user ask a question, the router determine which tool to use sql or rag or chart for anwsering questions.
User question
↓
router_chain chooses tool
↓
if SQL tool:
    run SQL function
    show chart
    sql_summary_chain explains result

if similar_violations:
    run RAG retrieval
    rag_summary_chain explains retrieved records

In [219]:
# LLM router template to determine which tool to use
router_template = """
You are a routing assistant for a food safety analytics system.

Choose exactly one tool:

top_violation_types
- Use for questions about the most common violation types or top violation descriptions.

violation_count_by_year_and_severity
- Use for questions about violation record trends over time, year-based trends, or trends split by severity.

top_violation_owner
- Use for questions about restaurants/businesses with repeated violations, high overall violation scores, high-risk counts, high-severity counts, serious outcomes, repeat offenders, or businesses with many violations.

similar_violations
- Use for semantic search questions asking for similar violations, related issues, corrective actions, examples, inspection comments, or violation patterns.


unknown
- Use if the question does not match any tool.

User question:
{text}

Return only the tool name.
"""

router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["text"]
)

router_chain = router_prompt | llm | StrOutputParser()

In [220]:
def infer_rank_by(question):
    q = question.lower()

    if "high severity" in q or "severe" in q:
        return "high_count"

    if "high risk" in q or "risk" in q:
        return "risk_count"

    if "most violations" in q or "many violations" in q or "repeat" in q:
        return "total_count"

    return "score"

In [179]:
#test router chain
tool1= router_chain.invoke({"text": "Which businesses had the most serious violations in 2024?"})
print(tool1)
tool2= router_chain.invoke({"text": "Find violations similar to sanitizer or dish machine problems."})
print(tool2)
tool3= router_chain.invoke({"text": "what is the violation trend looks like year after year for different severity level?"})
print(tool3)
tool4= router_chain.invoke({"text": "What are most common violations type in 2025"})
print(tool4)

top_violation_owner
similar_violations
violation_count_by_year_and_severity
top_violation_types


In [180]:
#build route question function
def route_question(question):
    route = router_chain.invoke({"text": question})
    return route

Build route function to route the action to the right place.

In [181]:
def summarize_sql(question, route, df, max_rows=10):
    result_text = df.head(max_rows).to_string(index=False)

    summary = sql_summary_chain.invoke({
        "question": question,
        "tool_name": route,
        "result": result_text
    })

    return summary

In [182]:
def summarize_rag(question, retrieved_df):
    context = format_rag_context(retrieved_df)

    summary = rag_chain.invoke({
        "question": question,
        "context": context
    })

    return summary

In [223]:
#build function to execute the route and return the anwser
def ask(question, year=None, limit=10, k=3, show_chart=True, chart_type="auto"):
    #route to route_question tool to determine which tool to use, output tool name
    route = route_question(question).strip()

    #if tool is top_violation_types, run top_violation_types sql tool
    if route == "top_violation_types":
        df = top_violation_types(year=year, limit=limit)
        #if show_chart is true, show chart
        chart = None
        if show_chart:
            #exctue chart_top_violation_types function
            chart = chart_top_violation_types(df)
        #use sql_summary function to summarize the result
        summary = summarize_sql(question, route, df)

        #return tool name, type of the tool (sql or rag), data details, summary of the result
        return {
            "route": route,
            "type": "sql",
            "data": df,
            "chart": chart,
            "summary": summary
        }
    #if tool is top_violation_owner, run top_violation_owner sql tool
    elif route == "top_violation_owner":
        rank_by = infer_rank_by(question)
        df = top_violation_owner(
        year=year,
        limit=limit,
        rank_by=rank_by
    )
        #if show_chart is true, show chart
        chart = None
        if chart_type == "count":
            chart = chart_top_violation_owner(df, year=year)

        elif chart_type == "risk_scatter":
            chart = chart_owner_risk_scatter(df, year=year)

        elif chart_type == "auto":
            if "risk" in question.lower() or "score" in question.lower() or "severity" in question.lower():
                chart = chart_owner_risk_scatter(df, year=year)
            else:
                chart = chart_top_violation_owner(df, year=year)
        summary = summarize_sql(question, route, df)

        #return tool name, type of the tool (sql or rag), data details, summary of the result
        return {
            "route": route,
            "type": "sql",
            "data": df,
            "chart": chart,
            "summary": summary
        }
    elif route == "violation_count_by_year_and_severity":
        df = violation_count_by_year_and_severity(start_year=year)
        #if show_chart is true, show chart
        chart = None
        if show_chart:
            chart = chart_violation_count_by_year_and_severity(df)
        summary = summarize_sql(question, route, df)
        return {
            "route": route,
            "type": "sql",
            "data": df,
            "chart": chart,
            "summary": summary
        }

    #if tool is similar_violations, run similar_violations rag tool
    elif route == "similar_violations":
        df = retrieve_similar_violations(question, k=k)
        #use summarize_rag function to summarize the result
        summary = summarize_rag(question, df)

        return {
            "route": route,
            "type": "rag",
            "data": df,
            "summary": summary
        }
    #if tool is not found, return error
    else:
        return {
            "route": route,
            "type": "error",
            "data": None,
            "summary": "No matching tool found."
        }

In [226]:
#test1
result = ask(
    "What are the top 10 businesses with the most violations in 2025?",
    year=2025,
    show_chart=True
)

print(result["summary"])
result["chart"].show()

1. Direct answer: The top 10 businesses with the most violations in 2025 are led by Dans Mini Dogs, followed by Boston Restaurant Bar & Grill and New York Pizza.

2. Key evidence from SQL result:
   - Dans Mini Dogs: 371 total violations
   - Boston Restaurant Bar & Grill: 126 total violations
   - New York Pizza: 105 total violations

3. Business interpretation: Dans Mini Dogs has significantly more violations than the other businesses, indicating potential systemic issues in compliance or operational practices.

4. Limitation: The SQL result only provides data for 2025 and does not include historical trends or comparisons to previous years.


In [106]:
query = """
SELECT
    businessname,
    COUNT(*) AS total_violations,
    SUM(CASE WHEN result_group IN (
        'fail',
        'extended_fail',
        'hearing',
        'temporary_suspension',
        'voluntary_closure_avoid',
        'forced_closure'
    ) THEN 1 ELSE 0 END) AS serious_outcome_count,
    AVG(severity_score) AS avg_severity
FROM inspections
WHERE businessname ILIKE '%Caffe Nero%'
  AND severity_score IS NOT NULL
  AND EXTRACT(year FROM resultdttm) = 2025
GROUP BY businessname
ORDER BY serious_outcome_count DESC, avg_severity DESC
"""

con.execute(query).df()

,businessname,total_violations,serious_outcome_count,avg_severity
0,Caffe Nero,218,150.0,1.40367
1,"Caffe Nero ""Express""",5,2.0,1.00000


In [105]:
question = "show me Caffe Nero restaurant violation in 2025?"
ask(question, year = 2025)

{'route': 'top_violation_owner',
 'type': 'sql',
 'data':                     businessname                  address  total_violations  \
 0                 Dans Mini Dogs   1010  MASSACHUSETTS AV               371   
 1  Boston Restaurant Bar & Grill           1251  RIVER ST               126   
 2                    City Winery              1  CANAL ST               101   
 3              Fritay Restaurant            532  RIVER ST                97   
 4                WIT Beatty Cafe       550  HUNTINGTON AV                93   
 5           Halal Indian Cuisine       736  HUNTINGTON AV                94   
 6                 New York Pizza  433   MASSACHUSETTS  AV               105   
 7                         Vaanga            102  WATER ST                90   
 8             BOS' Sichuan Taste          204  HARVARD AV                90   
 9                 La Mesa Market           744  DUDLEY ST                83   
 
    serious_outcome_count  avg_severity  
 0                 

In [ ]:
def show_result(result):
    print(f"Route: {result['route']}")
    print(f"Type: {result['type']}")
    print("\nSummary:\n")
    print(result["summary"])

    print("\nData:\n")
    display(result["data"])

9. Evaluation and lessions learnt


10. Final demo function

In [ ]:
import subprocess

process = subprocess.Popen(["streamlit", "run", "app.py"])

2026-05-10 20:38:12.505 Uvicorn server started on 0.0.0.0:8504



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8504
  Network URL: http://192.168.1.162:8504

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            


/Users/jiezhao/miniconda3/envs/llm_langchain/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


────────────────────────── Traceback (most recent call last) ───────────────────────────
  /Users/jiezhao/miniconda3/envs/llm_langchain/lib/python3.14/site-packages/streamlit/  
  runtime/scriptrunner/exec_code.py:129 in exec_func_with_error_handling                
                                                                                        
  /Users/jiezhao/miniconda3/envs/llm_langchain/lib/python3.14/site-packages/streamlit/  
  runtime/scriptrunner/script_runner.py:689 in code_to_exec                             
                                                                                        
  /Users/jiezhao/Documents/Harvard_Data_Sciense/CSCI-104/finalproject/Submit/app.py:2   
  in <module>                                                                           
                                                                                        
     1 import streamlit as st                                                           
  ❱  2 from backend i

Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.aria.image_processing_pil_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.auto.image_processing_auto`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.beit.image_processing_beit`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `__path__` from `.models.beit.image_processing_pil_beit`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
2026-05-10 20:38:18.029 Examining the path of transformers.models.beit.image_processing_pil_beit_fast raised:
Traceback (most recent call

: 

In [ ]:
import pickle
import faiss

#rag_df.to_pickle("data/rag_df.pkl")
#faiss.write_index(index, "data/faiss_index.index")

Demo:

In [85]:
schema_description = """
Table name: inspections

Columns:
- businessname: name of inspected business
- address: business address
- zip: ZIP code
- resultdttm: inspection datetime
- result: raw inspection result code
- result_group: normalized inspection result group
- inferreddescription: plain English meaning of inspection result
- violation: violation code
- violdesc: violation description
- viol_status: violation status
- severity_level: low, medium, high
- severity_score: numeric severity score
- comments: inspector comments

Important definitions:
- Serious outcomes include result_group in:
  ('fail', 'extended_fail', 'hearing', 'temporary_suspension',
   'voluntary_closure_avoid', 'forced_closure')
- Use EXTRACT(year FROM resultdttm) for year filtering.
- Use violdesc for violation type analysis.
"""

sql_generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert DuckDB SQL generator.

Generate one valid DuckDB SQL query for the user's question.
Use only the provided schema.
Do not invent columns.
Return only SQL. Do not include markdown fences.
"""),
    ("human", """
Schema:
{schema}

User question:
{question}
""")
])

sql_generation_chain = sql_generation_prompt | llm | StrOutputParser()

def generate_sql_from_question(question):
    sql = sql_generation_chain.invoke({
        "schema": schema_description,
        "question": question
    })

    sql = sql.strip()

    # Remove accidental code fences if model adds them
    sql = sql.replace("```sql", "").replace("```", "").strip()

    return sql

In [86]:
def run_llm_generated_sql(question):
    sql = generate_sql_from_question(question)

    # Basic guardrail: allow SELECT only
    if not sql.lower().startswith("select"):
        raise ValueError(f"Only SELECT queries are allowed. Generated SQL:\n{sql}")

    try:
        df = con.execute(sql).df()
        return {
            "question": question,
            "sql": sql,
            "data": df,
            "error": None
        }
    except Exception as e:
        return {
            "question": question,
            "sql": sql,
            "data": None,
            "error": str(e)
        }

In [88]:
baseline = run_llm_generated_sql(
    "Which businesses had the most serious inspection outcomes in 2025?"
)

print(baseline["sql"])
print(baseline["error"])
baseline["data"].head()

SELECT businessname, COUNT(*) AS serious_outcomes_count
FROM inspections
WHERE EXTRACT(year FROM resultdttm) = 2025
AND result_group IN ('fail', 'extended_fail', 'hearing', 'temporary_suspension', 'voluntary_closure_avoid', 'forced_closure')
GROUP BY businessname
ORDER BY serious_outcomes_count DESC;
None


,businessname,serious_outcomes_count
0,Dans Mini Dogs,327
1,Caffe Nero,155
2,Dunkin Donuts,136
3,Subway,121
4,Boston Restaurant Bar & Grill,117


ask("What are the most common violations?")
ask("Show violation trend over time")
ask("Find violations similar to refrigeration problems")

Reference

https://data.boston.gov/dataset/food-establishment-inspections
https://supermemory.ai/blog/best-open-source-embedding-models-benchmarked-and-ranked/